# Análisis exploratorio: BAF Base

**Objetivo.** Conocer los datos antes de modelar y fijar las decisiones de
preprocesamiento con evidencia. Las conclusiones se trasladan a
`docs/datos.md` y a la memoria del TFM.

**Contenido**
1. Carga de los datos
2. Prevalencia de fraude por mes y por partición
3. Valores ausentes: ¿la ausencia se relaciona con el fraude?
4. Deriva entre meses
5. Atributo protegido (`customer_age >= 50`)

## 1. Carga de los datos

Se carga `Base.csv` con las rutas de `configs/local.yaml`. `resolve_path`
convierte la ruta en absoluta desde la raíz del repositorio, para que funcione
aunque el notebook se ejecute desde `notebooks/`.

In [ ]:
import pandas as pd

from src.config import load_config, resolve_path
from src.data import load_data, MISSING_AS_MINUS_ONE

config = load_config(resolve_path("configs/local.yaml"))
df = load_data(config)

df.shape


(1000000, 32)

**Resultado.** 1.000.000 de filas y 32 columnas, como indica la documentación
del BAF. La verificación de tipos, rangos y ausentes codificados como negativos
está en `docs/datos.md`.

## 2. Prevalencia de fraude por mes y por partición

Partición fija del proyecto: entrenamiento = meses 0–4, validación = mes 5,
test = meses 6–7.

**Qué se mira.** El número de solicitudes y la tasa de fraude en cada mes y en
cada partición.

**Por qué importa.** El umbral se fija en validación (5 % de FPR) y se aplica
tal cual al test. Si la distribución cambia entre el mes 5 y los meses 6–7, ese
umbral no dará exactamente un 5 % de FPR en test.

In [5]:
import numpy as np

""""
def particionar_df(x):
    if x <= 4:
        return "train"
    elif x == 5:
        return "eval"
    elif 5 < x <=7:
        return "test"
    else:
        return 0

df["split"] = df["month"].apply(particionar_df)


train = df[df["split"].str.contains("train")].copy().reset_index(drop=True)
eval = df[df["split"].str.contains("eval")].copy().reset_index(drop=True)
test = df[df["split"].str.contains("test")].copy().reset_index(drop=True)

print(len(train))

"""
conditions = [df["month"] <=4, df["month"] ==5, df["month"] >= 6]
df["split"] = np.select(conditions, ["train", "val", "test"], default = "unknown")

assert(df["split"] != "unknown").all(), "Hay meses fuera de 0-7"

df.groupby("split")["fraud_bool"].agg(["size", "sum", "mean"])



,size,sum,mean
split,,,
test,205011,2878,0.014038
train,675666,6740,0.009975
val,119323,1411,0.011825


In [4]:
df.groupby("month")["fraud_bool"].agg(["size", "sum", "mean"])

,size,sum,mean
month,,,
0,132440,1500,0.011326
1,127620,1198,0.009387
2,136979,1198,0.008746
3,150936,1392,0.009222
4,127691,1452,0.011371
5,119323,1411,0.011825
6,108168,1450,0.013405
7,96843,1428,0.014746


## 3. Valores ausentes: ¿la ausencia se relaciona con el fraude?

**Qué se mira.**
- En las columnas con -1 como ausente, la tasa de fraude con y sin el dato.
- En `intended_balcon_amount`, la tasa de fraude por tramos de los valores
  negativos, para decidir si son ausentes (decisión pendiente en
  `docs/datos.md`).

**Por qué importa.** Si la ausencia predice el fraude, el indicador de
ausencia que usarán las redes aporta información.

In [12]:
df_minus_one = df[MISSING_AS_MINUS_ONE]

assert(df_minus_one >= -1).all().all(), "Hay valores distintos de -1"

In [15]:
rows = []

for col in MISSING_AS_MINUS_ONE:
    is_missing = (df[col] == -1)
    rows.append({
        "column": col,
        "n_missing": is_missing.sum(),
        "pct_missing": is_missing.mean(),
        "fraud_rate_missing": df.loc[is_missing, "fraud_bool"].mean(),
        "fraud_rate_present": df.loc[~is_missing, "fraud_bool"].mean(),
        "ratio": df.loc[is_missing, "fraud_bool"].mean() / df.loc[~is_missing, "fraud_bool"].mean()
    })


summary = pd.DataFrame(rows)
summary

,column,n_missing,pct_missing,fraud_rate_missing,fraud_rate_present,ratio
0,prev_address_months_count,712920,0.712920,0.014215,0.003118,4.559529
1,current_address_months_count,4254,0.004254,0.003291,0.011062,0.297505
2,bank_months_count,253635,0.253635,0.016323,0.009230,1.768423
3,session_length_in_minutes,2015,0.002015,0.008933,0.011033,0.809645
4,device_distinct_emails_8w,359,0.000359,0.011142,0.011029,1.010255


## 4. Deriva entre meses

**Qué se mira.** Si la distribución de las variables cambia de un mes a otro,
sobre todo entre entrenamiento (0–4) y test (6–7).

**Por qué importa.** El modelo se entrena con el pasado y se evalúa con el
futuro. Una deriva fuerte explicaría caídas de rendimiento en test.

In [20]:
num_cols = df.select_dtypes(include = "number").columns

df.groupby("month")[num_cols].mean()

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,...,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,session_length_in_minutes,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
month,,,,,,,,,,,,,,,,,,,,,
0,0.011326,0.545999,0.480271,17.995500,85.359333,33.566898,0.721744,10.320819,2168.830384,7436.980252,...,0.896927,12.224049,0.170757,591.471685,0.039709,8.216170,0.677582,1.025619,0.0,0.0
1,0.009387,0.520698,0.526254,13.645886,100.431257,33.093637,1.346258,8.377181,1676.173719,6821.410104,...,0.859959,10.662592,0.208776,522.967247,0.020757,8.092347,0.590644,1.034117,0.0,1.0
2,0.008746,0.496552,0.536123,18.444623,80.539418,33.664503,1.511642,9.040804,1713.588492,6465.410441,...,0.872083,10.405215,0.175195,451.502055,0.022281,8.844311,0.559867,1.034159,0.0,2.0
3,0.009222,0.563151,0.485859,20.081637,79.237511,34.426711,0.996418,10.338422,1541.099744,5960.157365,...,0.889947,10.503180,0.272321,589.499324,0.022685,7.473184,0.520466,1.009706,0.0,3.0
4,0.011371,0.580703,0.490350,16.521697,90.412261,33.537916,1.141969,7.247711,1561.108817,5323.332006,...,0.880493,10.702712,0.244301,507.618235,0.026556,7.038164,0.480512,1.010635,0.0,4.0
5,0.011825,0.580014,0.473809,9.938302,95.400208,33.557068,0.639925,7.723208,1418.399479,4755.840267,...,0.912816,11.089497,0.274147,478.774168,0.024203,6.593297,0.592484,1.010769,0.0,5.0
6,0.013405,0.626423,0.462752,15.095010,89.784437,34.417388,0.860065,9.575181,1295.843928,3958.599974,...,0.913819,10.970990,0.240210,489.143370,0.022983,7.039296,0.570945,1.008884,0.0,6.0
7,0.014746,0.617458,0.484813,21.765652,70.565389,33.074461,0.884328,5.616526,985.613013,3605.322602,...,0.900013,10.041035,0.193488,465.658230,0.021612,6.585083,0.648173,1.008436,0.0,7.0


## 5. Atributo protegido: edad

Grupos: `customer_age >= 50` frente a `< 50`, el corte del código oficial del
BAF (ver `docs/datos.md`).

**Qué se mira.** El tamaño de cada grupo y su tasa de fraude, en total y por
partición.

**Por qué importa.** La métrica de equidad es el cociente de FPR entre los
dos grupos. Un grupo pequeño en test da un FPR con mucha varianza.